Input file location: /content/sample_data/example_label_100.h5

In [ ]:
!pip uninstall -y torch torchvision torchaudio torch_xla


Found existing installation: torch 2.6.0
Uninstalling torch-2.6.0:
  Successfully uninstalled torch-2.6.0
Found existing installation: torchvision 0.21.0
Uninstalling torchvision-0.21.0:
  Successfully uninstalled torchvision-0.21.0
Found existing installation: torch_xla 2.8.0
Uninstalling torch_xla-2.8.0:
  Successfully uninstalled torch_xla-2.8.0


In [ ]:
!pip install torch==2.6.0 \
  torch_xla[tpu] -f https://storage.googleapis.com/tpu-pytorch/wheels/tpuvm/torch_xla-2.6.0.html


Looking in links: https://storage.googleapis.com/tpu-pytorch/wheels/tpuvm/torch_xla-2.6.0.html
  Using cached torch-2.6.0-cp312-cp312-manylinux1_x86_64.whl.metadata (28 kB)
  Using cached torch_xla-2.8.0-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (16 kB)
Using cached torch-2.6.0-cp312-cp312-manylinux1_x86_64.whl (766.6 MB)
Using cached torch_xla-2.8.0-cp312-cp312-manylinux_2_28_x86_64.whl (88.9 MB)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
fastai 2.8.3 requires torchvision>=0.11, which is not installed.


In [ ]:
!pip uninstall -y torch torchvision torch_xla
!pip install torch==2.6.0 torchvision==0.21.0 \
  torch_xla[tpu] -f https://storage.googleapis.com/tpu-pytorch/wheels/tpuvm/torch_xla-2.6.0.html


Found existing installation: torch 2.6.0
Uninstalling torch-2.6.0:
  Successfully uninstalled torch-2.6.0
Found existing installation: torch_xla 2.8.0
Uninstalling torch_xla-2.8.0:
  Successfully uninstalled torch_xla-2.8.0
Looking in links: https://storage.googleapis.com/tpu-pytorch/wheels/tpuvm/torch_xla-2.6.0.html
  Using cached torch-2.6.0-cp312-cp312-manylinux1_x86_64.whl.metadata (28 kB)
  Using cached torchvision-0.21.0-cp312-cp312-manylinux1_x86_64.whl.metadata (6.1 kB)
  Using cached torch_xla-2.8.0-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (16 kB)
Using cached torch-2.6.0-cp312-cp312-manylinux1_x86_64.whl (766.6 MB)
Using cached torchvision-0.21.0-cp312-cp312-manylinux1_x86_64.whl (7.2 MB)
Using cached torch_xla-2.8.0-cp312-cp312-manylinux_2_28_x86_64.whl (88.9 MB)


In [ ]:
!pip install awkward

In [ ]:
!pip install spconv-cu118  # For PyTorch with CUDA 11.8


In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '-1'

In [ ]:
import h5py

with h5py.File('/content/sample_data/example_label_100.h5', 'r') as f1, \
     h5py.File('/content/sample_data/example_xyze_100.h5', 'r') as f2:

    label_data = f1[list(f1.keys())[0]][:]
    xyze_data  = f2[list(f2.keys())[0]][:]

print(label_data.shape, xyze_data.shape)


(100,) (100,)


In [ ]:
#import h5py
import numpy as np
import awkward as ak
import matplotlib.pyplot as plt
from tqdm import tqdm

import torch
from torch import nn
import spconv.pytorch as spconv

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cpu


Opening h5 file

In [ ]:
#with h5py.File('data/train_xyze_1e4.h5', 'r') as f:
with h5py.File('content/sample_data/train_xyze_1e4.h5', 'r') as f:
    # Access the dataset
    label_events = f['data']

    val_data = []

    data = []
    for i, event in enumerate(label_events):
        if i < 500:
            val_data.append(f['data'][i].reshape(-1,4))
            continue

        pts = f['data'][i].reshape(-1,4)
        data.append(pts)

FileNotFoundError: [Errno 2] Unable to synchronously open file (unable to open file: name = 'data/train_xyze_1e4.h5', errno = 2, error message = 'No such file or directory', flags = 0, o_flags = 0)

In [ ]:
# Open the labels file
#with h5py.File('data/train_label_1e4.h5', 'r') as f:
with h5py.File('content/sample_data/train_label_1e4.h5', 'r') as f:
    # Access the dataset
    label_events = f['labels']

    labels = []
    val_labels = []

    for i, event in enumerate(label_events):
        if i < 500:
            val_labels.append(event)
            continue

        labels.append(event)

In [ ]:
# Define the training truth labels
# 2 is the label for Michel electrons
truth = [(event_labels == 2).astype(np.int32) for event_labels in labels]
val_truth = [(event_labels == 2).astype(np.int32) for event_labels in val_labels]

In [ ]:
# Input: sparse tensor with coordinates and features
# coordinates: Nx4 tensor (batch_idx, x, y, z)
# features: Nx1 tensor (voxel values)
# Iterate over the events and create sparse tensors
# Example input data

features = []
coords = []

# Iterate over all the events and create sparse tensors that are then merged
# For each input_coords, add another column for the batch index, which is just the index of the event
for i, input_event in enumerate(data):
    # Use the first event for demonstration
    # input_event = data[0]
    # Use the first three columns as coordinates
    input_coords = torch.tensor(input_event[:, :3], dtype=torch.int32)
    input_coords = torch.cat((torch.full((input_coords.shape[0], 1), i, dtype=torch.int32), input_coords), dim=1)  # Add batch index
    # Use the fourth column as features
    input_feats = torch.tensor(input_event[:, 3], dtype=torch.float32).unsqueeze(1)

    features.append(input_feats)
    coords.append(input_coords)

val_features = []
val_coords = []
for i, input_event in enumerate(val_data):
    # Use the first event for demonstration
    # input_event = data[0]
    # Use the first three columns as coordinates
    input_coords = torch.tensor(input_event[:, :3], dtype=torch.int32)
    input_coords = torch.cat((torch.full((input_coords.shape[0], 1), i, dtype=torch.int32), input_coords), dim=1)  # Add batch index
    # Use the fourth column as features
    input_feats = torch.tensor(input_event[:, 3], dtype=torch.float32).unsqueeze(1)

    val_features.append(input_feats)
    val_coords.append(input_coords)

In [ ]:
input_feats = torch.cat(features, dim=0).to(device)
input_coords = torch.cat(coords, dim=0).to(device)

val_input_feats = torch.cat(val_features, dim=0).to(device)
val_input_coords = torch.cat(val_coords, dim=0).to(device)

In [ ]:
class AutoEncoder(nn.Module):
    def __init__(self, shape):
        super().__init__()
        self.encoder = spconv.SparseSequential(
            # TODO try submanifold convolution
            spconv.SparseConv3d(1, 1, 3, 1, algo=spconv.ConvAlgo.Native, indice_key="cp0"),
            spconv.SparseConv3d(1, 1, 3, 1, algo=spconv.ConvAlgo.Native, indice_key="cp1"),
            # TODO maxpooling
            # spconv.SparseMaxPool3d(kernel_size=2, stride=2, padding=0),
            # TODO batch normalization
            nn.ReLU(),
            spconv.SparseBatchNorm(1),
            spconv.SparseConv3d(1, 1, 3, 1, algo=spconv.ConvAlgo.Native, indice_key="cp2"),
            spconv.SparseConv3d(1, 1, 3, 1, algo=spconv.ConvAlgo.Native, indice_key="cp3"),
            # spconv.SparseMaxPool3d(kernel_size=2, stride=2, padding=0),
            nn.ReLU(),
            spconv.SparseBatchNorm(1),
        ).to(device)

        self.decoder = spconv.SparseSequential(
            spconv.SparseInverseConv3d(1, 1, 3, algo=spconv.ConvAlgo.Native, indice_key="cp3"),
            spconv.SparseInverseConv3d(1, 1, 3, algo=spconv.ConvAlgo.Native, indice_key="cp2"),
            nn.ReLU(),
            spconv.SparseBatchNorm(1),
            # spconv.SparseMaxPool3d(kernel_size=2, stride=2, padding=0),
            # nn.ReLU(),
            spconv.SparseInverseConv3d(1, 1, 3, algo=spconv.ConvAlgo.Native, indice_key="cp1"),
            spconv.SparseInverseConv3d(1, 1, 3, algo=spconv.ConvAlgo.Native, indice_key="cp0"),
        )
        self.shape = shape

    def forward(self, features, coors, batch_size):
        coors = coors.int()
        x = spconv.SparseConvTensor(features, coors, self.shape, batch_size)
        # print("Input shape:", x)
        x = self.encoder(x)
        x = self.decoder(x)
        return x

In [ ]:
# TODO
def compute_validation_output(model, val_truth, val_input_feats, val_input_coords):
    """Compute the model output for the validation data"""
    val_batch_size = 100

    val_n_voxels = [len(e) for e in val_truth]
    val_n_voxels_cumsum = np.cumsum(val_n_voxels)

    preds = []

    for i, batch in enumerate(range(0, len(val_truth), val_batch_size)):
        voxel_i_lower = 0 if batch == 0 else val_n_voxels_cumsum[batch - 1]
        voxel_i_upper = val_n_voxels_cumsum[batch + val_batch_size-1] if (batch + val_batch_size-1) < len(val_truth) else val_n_voxels_cumsum[-1]
        batch_input_feats = val_input_feats[voxel_i_lower:voxel_i_upper]
        batch_input_coords = val_input_coords[voxel_i_lower:voxel_i_upper].detach().clone()

        # From batch_input_coords, subtract the batch number to make indices start from 0
        batch_input_coords[:, 0] = batch_input_coords[:, 0] - batch
        # Defensive: skip if batch_input_coords is empty
        if batch_input_coords.shape[0] == 0:
            print("Empty batch. Skip")
            continue

        # Defensive: check max batch index only if not empty
        # if batch_input_coords.shape[0] > 0:
            # assert batch_input_coords[:, 0].max().item() == val_batch_size - 1, batch_input_coords[:, 0].max().item()
            # assert batch_input_coords[:, 0].min().item() == 0, batch_input_coords[:, 0].min().item()

        model.eval()
        with torch.no_grad():
            output_sparse = model(batch_input_feats, batch_input_coords, batch_size=val_batch_size if batch + val_batch_size < len(data) else len(data) - batch)

            preds.append(output_sparse.features)

    preds = torch.cat(preds, dim=0)

    return preds

In [ ]:
# train model using the truth labels
n_voxels = ak.num(truth)
n_voxels_cumsum = np.cumsum(n_voxels)

n_epochs = 1
loss_per_epoch = np.zeros(n_epochs)
val_loss_per_epoch = np.zeros(n_epochs)

loss_fn = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

batch_size = 5  # Define batch size
for epoch in range(n_epochs):  # Example training loop
    batch_indices = range(0, len(data), batch_size)
    loss_per_batch = np.zeros(len(batch_indices))

    for i, batch in enumerate(tqdm(batch_indices)):  # Batch size of 5
        voxel_i_lower = 0 if batch == 0 else n_voxels_cumsum[batch - 1]
        voxel_i_upper = n_voxels_cumsum[batch + batch_size-1] if (batch + batch_size-1) < len(data) else n_voxels_cumsum[-1]

        batch_input_feats = input_feats[voxel_i_lower:voxel_i_upper].to(device)
        batch_input_coords = input_coords[voxel_i_lower:voxel_i_upper].detach().clone().to(device)
        # From batch_input_coords, subtract the batch number to make indices start from 0
        batch_input_coords[:, 0] = batch_input_coords[:, 0] - batch

        # Defensive: skip if batch_input_coords is empty
        if batch_input_coords.shape[0] == 0:
            print("Empty batch. Skip")
            continue

        # Defensive: check max batch index only if not empty
        if batch_input_coords.shape[0] > 0:
            assert batch_input_coords[:, 0].max().item() == batch_size - 1, batch_input_coords[:, 0].max().item()
            assert batch_input_coords[:, 0].min().item() == 0, batch_input_coords[:, 0].min().item()

        batch_truth = torch.tensor(ak.flatten(truth[batch:batch+batch_size]), dtype=torch.float32).unsqueeze(1).to(device)

        model.train()
        optimizer.zero_grad()
        output_sparse = model(batch_input_feats, batch_input_coords, batch_size=batch_size)
        loss = loss_fn(output_sparse.features.to(device), batch_truth.to(device))
        loss.backward()
        optimizer.step()
        loss_per_batch[i] = loss.detach().clone().item()

    # TODO Calculate the loss for validation data
    model.eval()
    # output_sparse = model(val_input_feats, val_input_coords, batch_size=val_input_feats[:, 0].max())
    # val_loss_per_epoch[epoch] = loss_fn(output_sparse, val_truth)
    # Save the loss
    loss_per_epoch[epoch] = np.mean(loss_per_batch)

In [ ]:
plt.plot(range(n_epochs), loss_per_epoch)
plt.plot(range(n_epochs), val_loss_per_epoch)
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.grid()

In [ ]:
# Evaluate the performance of the model
model.eval()
output_sparse = model(val_input_feats, val_input_coords, batch_size=val_input_feats[:, 0].max())
# Output values:
model_output = output_sparse.features

In [ ]:
# Calculate the F1 score:
from sklearn.metrics import f1_score
cut = 0.5
model_binary = []
for n in model_output: # assumes model_output is a simple list
  if n < 0.5: model_binary.append(0)
  if n >= 0.5: model_binary.append(1)
all_truth = [a for t in truth for a in t]
f1_binary = f1_score(all_truth, model_binary, average='binary', pos_label=1)
print(f"F1 score: {f1_binary}")

In [ ]:
# plot
f1_scores = []
cuts = np.arange(0,1,10)
for c in cuts:
  model_binary = []
  for n in model_output: # assumes model_output is a simple list
    if n < c: model_binary.append(0)
    if n >= c: model_binary.append(1)
  f1_scores.append(f1_score(all_truth, model_binary, average='binary', pos_label=1))
plt.scatter(cuts, f1_scores)
plt.xlabel("Cut")
plt.ylabel("F1 Score")
plt.show()

TODO

* F1 score calculator

* Implement the sparse autoencoder

* Implement an alternative model (e.g. U-Net, graph neural network, TorchPoint3D)

* Validation plot: the evolution of the loss during the training of the model, for both training and validation data

* Validation plot: the F1 score as a function of the cut threshold of the neural network output. This will show where should put our threshold for classifying a voxel as Michel or not to get the optimal F1 score. One can also potentially show the efficiency, purity, and histograms of the background / Michel electrons in the same plot

# Graph neural netwwork

In [ ]:
with h5py.File('data/train_xyze_1e4.h5', 'r') as f:
    # Access the dataset
    label_events = f['data']
    data = []
    for i, event in enumerate(label_events):
        pts = f['data'][i].reshape(-1,4)
        data.append(pts)


In [ ]:
# Open the labels file
with h5py.File('data/train_label_1e4.h5', 'r') as f:
    # Access the dataset
    label_events = f['labels']

    labels = []

    for i, event in enumerate(label_events):
        labels.append(event)

truth = [(event_labels == 2).astype(np.int32) for event_labels in labels]

## Generate graphs from the data
This uses a k-nearest neighbour to create graphs. Each node has the feature corresponding to the energy, and each edge between nodes has a feature corresponding to the distance between the two voxels.

The data is also split into training (9000 events), testing (500 events) and validation (500 events) data.

In [ ]:
train_data = []
val_data = []
test_data = []

# Iterate over all the events and create sparse tensors that are then merged
# For each input_coords, add another column for the batch index, which is just the index of the event
for i, (event_coords, input_labels) in enumerate(zip(tqdm(data), truth)):
    # Use the first three columns as coordinates
    input_coords = torch.tensor(event_coords[:, :3], dtype=torch.int32)
    # Use the fourth column as features
    input_features = torch.tensor(event_coords[:, 3], dtype=torch.float32).unsqueeze(1)

    edge_index = knn_graph(input_coords, k=4, loop=False)
    # Compute the edge attributes (distances)
    row, col = edge_index
    edge_attr = input_coords[row] - input_coords[col]
    edge_attr = torch.norm(edge_attr.to(torch.float32), p=2, dim=1).view(-1, 1)  # Reshape to [num_edges, 1]
    # edge_index = radius_graph(input_coords, r=3.0, loop=False)

    event_data = Data(
        x=input_features,
        edge_index=edge_index,
        y=torch.tensor(input_labels, dtype=torch.float32),
        edge_attr=edge_attr,
    )

    if i < 500:
        val_data.append(event_data)
        continue
    if i < 1000:
        test_data.append(event_data)
        continue

    train_data.append(event_data)

In [ ]:
# Create DataLoader objects which can handle batching of the data
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
val_loader = DataLoader(val_data, batch_size=32, shuffle=False)
val_loader = DataLoader(test_data, batch_size=32, shuffle=False)

In [ ]:
# Count the number of nodes in the training and validation data
# This information is used for calculating the average loss per voxel during training
train_n_nodes = sum(len(d.x) for d in train_loader)
val_n_nodes = sum(len(d.x) for d in val_loader)
print(f"Train nodes: {train_n_nodes}, Val nodes: {val_n_nodes}")

## Define the graph neural network

In [ ]:
class GNN(torch.nn.Module):
    def __init__(self, hidden_dim, output_dim, heads=1):
        super(GNN, self).__init__()
        self.conv1 = GATConv(in_channels=1, out_channels=hidden_dim, heads=heads, edge_dim=1)
        self.conv2 = GATConv(in_channels=hidden_dim * heads, out_channels=hidden_dim, heads=1, edge_dim=1)
        self.conv3 = GATConv(in_channels=hidden_dim, out_channels=output_dim, heads=1, concat=False, edge_dim=1)

    def forward(self, x, edge_index, edge_attr):
        # Layer 1
        x = F.elu(self.conv1(x, edge_index, edge_attr))
        # Layer 2
        x = F.elu(self.conv2(x, edge_index, edge_attr))
        # Output layer
        x = self.conv3(x, edge_index, edge_attr)
        return x


## Train the GNN

In [ ]:
# Initialize model
model = GNN(hidden_dim=4, output_dim=1).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=5e-4)

train_loss_list = []
val_loss_list = []

# Training loop with validation
best_val_f1 = 0.0
# Run 50 epochs
for epoch in tqdm(range(50)):
    # Training
    model.train()
    train_loss = 0.0
    for batch in train_loader:
        optimizer.zero_grad()
        out = model(batch.x.to(device), batch.edge_index.to(device), batch.edge_attr.to(device))
        # weight = batch.y.unsqueeze(1) + 0.1
        loss = F.binary_cross_entropy_with_logits(out, batch.y.unsqueeze(1).to(device), reduction="sum")
        # loss = (loss * weight).sum()
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    # Divide by the total number of nodes in all batches
    train_loss /= train_n_nodes

    train_loss_list.append(train_loss)

    # Validation
    model.eval()
    val_loss = 0.0
    preds = []

    with torch.no_grad():
        for batch in val_loader:
            # weight = batch.y.unsqueeze(1) + 0.1
            out = model(batch.x.to(device), batch.edge_index.to(device), batch.edge_attr.to(device))
            loss = F.binary_cross_entropy_with_logits(out, batch.y.unsqueeze(1).to(device), reduction="sum")
            # loss = (loss * weight).sum()
            preds.append(out)
            val_loss += loss.item()

    val_loss /= val_n_nodes
    val_loss_list.append(val_loss)


    print(
        f"Epoch {epoch+1}, Train Loss: {train_loss:.4f}, "
        f"Val Loss: {val_loss:.4f}, "
    )
    # Save intermediate states of the model
    torch.save(model.state_dict(), f"model_epoch_{epoch+1}.pt")

In [ ]:
# Plot the training and validation loss
fig, ax = plt.subplots(layout='constrained')
ax.plot(train_loss_list, label='Training data loss', lw=3, color=plt.cm.viridis(0.2))
ax.plot(val_loss_list, label='Validation data loss', lw=3, color=plt.cm.viridis(0.6))
ax.set(xlabel='Epoch', ylabel='Loss')
ax.legend()
ax.grid()

## Plot the results

In [ ]:
# In case your session stopped, uncomment and run this

# model = GNN(hidden_dim=4, output_dim=1).to(device)
# selected_epoch = 20 # Change this value
# model.load_state_dict(torch.load(f"model_epoch_{selected_epoch}.pt"))

In [ ]:
# Get the model predictions on the validation data
model.eval()
preds = []

with torch.no_grad():
    for batch in val_loader:
        weight = batch.y.unsqueeze(1) + 0.1
        out = model(batch.x.to(device), batch.edge_index.to(device), batch.edge_attr.to(device))
        preds.append(out)


In [ ]:
# Plot the distribution of model outputs
preds_only_truth = torch.cat([F.sigmoid(batch)[batch2.y == 1] for batch2, batch in zip(val_loader, preds)]).squeeze()
preds_only_bkg = torch.cat([F.sigmoid(batch)[batch2.y == 0] for batch2, batch in zip(val_loader, preds)]).squeeze()
# plot the distribution of predictions

bins = np.linspace(0., 0.1, 100)

fig, ax = plt.subplots(figsize=(6, 4),
                       layout="constrained")
ax.hist(
    preds_only_bkg.cpu().numpy(),
    density=True, bins=bins, label="Background voxel predictions", color=plt.cm.viridis(0.2), linewidth=3, histtype="step",
)
# plot the distribution of predictions for only nodes with label 1
ax.hist(
    preds_only_truth.cpu().numpy(),
    density=True, bins=bins, label="Michel electron voxels", color=plt.cm.viridis(0.6), histtype="step", linewidth=3,
)

ax.legend()
ax.grid()
# ax.xlim(0, 0.1)
ax.set(xlabel='Predicted Probability', ylabel='Density', title='Distribution of GNN predictions', yscale="log", xlim=(0.013, 0.0825))

In [ ]:
preds_flat = F.sigmoid(torch.cat(preds).squeeze())

# Calculate the F1 score as a function of the cut value
cut_values = np.linspace(0., 0.08, 20)
f1_scores = []
for cut in cut_values:
    preds_cut = preds_flat > cut
    f1 = f1_score(
        torch.cat([batch.y for batch in val_loader]).cpu().numpy(),
        preds_cut.cpu().numpy(),
    )
    f1_scores.append(f1)

In [ ]:
# Plot the F1 score as a function of the cut value
fig, ax = plt.subplots(layout="constrained")
ax.plot(cut_values, f1_scores, color=plt.cm.viridis(0.4), lw=3)
ax.set(xlabel='Cut value', ylabel='$F_1$ score', ylim=(0, 0.09), xlim=(0, 0.08))
best_cut_value = cut_values[np.argmax(f1_scores)]
print(f"Best cut value: {best_cut_value}, Best F1 score: {max(f1_scores)}")
ax.axvline(best_cut_value, color=plt.cm.viridis(0.), linestyle='--', label=f'Best cut value: {best_cut_value:.4f}')

ax.grid()

fig.savefig("f1_score_vs_cut.svg")

In [ ]:
# Compute the final F1 score on the test data
model.eval()
test_loader = DataLoader(test_data, batch_size=32, shuffle=False)
preds = []
with torch.no_grad():
    for batch in test_loader:
        out = model(batch.x.to(device), batch.edge_index.to(device), batch.edge_attr.to(device))
        preds.append(out)

preds_flat = F.sigmoid(torch.cat(preds).squeeze())

# Print the final F1 score on the test data:
f1_score(torch.cat([batch.y for batch in test_loader]).cpu().numpy(), (preds_flat > best_cut_value).cpu().numpy())

In [ ]:
fig, ax = plt.subplots(layout="constrained")
ax.plot(cut_values, f1_scores, label='F1 Score')
ax.set_xlabel('Cut Value')
ax.set_ylabel('F1 Score')
ax.grid()